# Étude d'ablation — échelle et baselines de référence

Ce notebook **n'exécute aucune simulation** : il lit les artefacts d'une
campagne déjà produite.

```bash
python main.py run --config experiments/ablation.yaml
```

Il répond à deux questions distinctes, lues sur le **même** monde (même graine,
même grille, même flotte, mêmes tirages de comportement).

**1. L'écart entre Nearest et BRAM-EV Full vient de quel composant ?**
L'échelle n'ajoute qu'un composant à la fois, donc l'écart entre deux barreaux
consécutifs *est* la contribution du composant ajouté.

| Configuration | Méthode | Multi-stations | Réputation | Adaptation |
| --- | --- | :-: | :-: | :-: |
| Nearest | `greedy` | non | non | non |
| Multi-Station Only | `multistation` | oui | non | non |
| Multi-Station + Reputation | `multistation_rep` | oui | oui | non |
| BRAM-EV Full | `bramev` | oui | oui | oui |

**2. BRAM-EV fait-il mieux que des politiques de choix simples ?**
Les trois baselines partagent exactement le protocole de `multistation` —
diffusion aux stations du rayon de recherche, sans réputation ni adaptation —
et n'en diffèrent que par la **règle de sélection de l'offre**. À périmètre
d'information identique, un écart mesuré est donc imputable à la règle seule.

| Baseline | Méthode | Règle de choix |
| --- | --- | --- |
| Minimum Waiting Time | `min_waiting` | attente la plus faible |
| Load-Aware | `load_aware` | station la moins chargée à venir |
| Random Feasible | `random_feasible` | tirage uniforme parmi les offres reçues |

`greedy` sert aussi de baseline mono-station : c'est la seule méthode qui ne
contacte qu'une station.

Le plan est déclaré une seule fois, dans `src/experiments/methods.py` — les
variantes de BRAM-EV sont traitées par `ablation_variants.ipynb`.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import numpy as np
import pandas as pd
from IPython.display import Image, display

import src.experiments.methods as methods
from src.pipeline import ablation, figures
from src.pipeline.store import RunStore

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## Choix du run

Une campagne ne porte pas forcément les quatre barreaux :
`latest_with_methods` prend le run le plus récent qui les contient tous, et dit
ce que contiennent les autres s'il n'en trouve aucun. Pour cibler un run précis :
`RunStore.open('../results_grid/<run>')`.

In [ ]:
for path in RunStore.list_runs('../results_grid'):
    present = sorted({row['method'] for row in RunStore(path).read_summary()})
    print(f"{path.name}\n    {', '.join(present) or 'aucun cas'}")

In [ ]:
store = RunStore.latest_with_methods(methods.LADDER, '../results_grid')
params = store.read_params()
manifest = store.read_manifest()

print(store.root)
print(params.describe())
print(f"graine={params.seed} | commit={manifest['git_commit']} | "
      f"cas={manifest['nb_cases_done']}/{manifest['nb_cases_planned']}")

## Ce qui est réellement activé

Premier contrôle, avant toute lecture de résultat : les drapeaux effectivement
appliqués. Ils voyagent depuis le registre jusqu'à `summary.csv`
(`src/pipeline/tables.py`), donc cette table dit ce que la campagne a fait —
pas ce qu'elle était censée faire.

Un barreau qui ne bascule pas exactement un composant invaliderait toute
l'attribution des gains.

In [ ]:
summary = pd.read_csv(store.summary_path)

ORDRE = list(methods.LADDER) + list(methods.BASELINES)
summary['method'] = pd.Categorical(summary['method'], ORDRE + [
    m for m in summary['method'].unique() if m not in ORDRE], ordered=True)

ladder = summary[summary['method'].isin(methods.LADDER)].copy()
baselines = summary[summary['method'].isin(methods.BASELINES)].copy()
compare = summary[summary['method'].isin(ORDRE)].copy()

plan = (compare[['method', 'method_label', 'method_family', 'broadcast',
                 'reputation', 'adaptation', 'offer_choice', 'alpha_mode',
                 'reputation_scope', 'score_weighting']]
        .drop_duplicates()
        .sort_values('method')
        .set_index('method'))
plan

In [ ]:
# Un seul composant change d'un barreau au suivant, et les mécanismes internes
# restent ceux de BRAM-EV tout au long : sinon un barreau mélangerait deux effets.
composants = ['broadcast', 'reputation', 'adaptation']
mecanismes = ['offer_choice', 'alpha_mode', 'reputation_scope', 'score_weighting']

for avant, apres, libelle in methods.LADDER_STEPS:
    a, b = plan.loc[avant], plan.loc[apres]
    change = [c for c in composants if a[c] != b[c]]
    derive = [m for m in mecanismes if a[m] != b[m]]
    etat = 'OK' if (len(change) == 1 and not derive) else 'ANOMALIE'
    print(f"{etat:9} {avant:18} -> {apres:18} {libelle:26} "
          f"change={change} mécanismes_dérivés={derive}")

# Une baseline ne doit différer de `multistation` que par sa règle de choix :
# même diffusion, même rayon, ni réputation ni adaptation. Sinon l'écart mesuré
# ne serait plus imputable à la règle.
print()
if 'multistation' in plan.index:
    ref = plan.loc['multistation']
    partages = composants + ['alpha_mode', 'reputation_scope', 'score_weighting']
    for nom in methods.BASELINES:
        if nom not in plan.index:
            continue
        b = plan.loc[nom]
        derive = [c for c in partages if ref[c] != b[c]]
        etat = 'OK' if not derive else 'ANOMALIE'
        print(f"{etat:9} {nom:18} règle={b['offer_choice']:<9} "
              f"écarts_hors_règle={derive}")

In [ ]:
# Décomposition calculée à la volée depuis summary.csv. Le pipeline persiste
# exactement les mêmes tables (`ablation.csv`, `ablation_mean.csv`) et les
# réécrit à chaque cas ; les recalculer ici rend le notebook utilisable sur une
# campagne encore en cours, interrompue, ou antérieure à l'étude d'ablation.
detail = pd.DataFrame(ablation.detail_rows(summary.to_dict('records')))
moyennes = pd.DataFrame(ablation.mean_rows(detail.to_dict('records')))

print(f"{len(detail)} écarts calculés sur "
      f"{detail[['scenario', 'nb_cars']].drop_duplicates().shape[0]} mondes")

## Les méthodes côte à côte

Les quatre barreaux de l'échelle, puis les trois baselines de référence.

In [ ]:
METRIQUES = ['exact_satisfaction', 'rate_abs', 'mean_service_rate',
             'slot_waste_rate', 'nb_reservations', 'mean_waiting_time_min',
             'mean_offers_per_demand', 'total_ms_mean']

niveaux = ladder.pivot_table(index=['scenario', 'nb_cars'], columns='method',
                             values=METRIQUES, observed=True)
niveaux

In [ ]:
# Moyenne sur tous les mondes du run : une ligne par méthode comparée.
# L'échelle d'abord, les baselines ensuite — l'ordre de `ORDRE`.
(compare.groupby('method', observed=True)[METRIQUES]
        .mean()
        .rename(index=methods.label)
        .round(4))

## Contribution de chaque composant

`ablation_mean.csv` est écrit par le pipeline à chaque campagne
(`src/pipeline/ablation.py`). Chaque ligne est un couple (composant, métrique) :

* `mean_delta` / `mean_delta_pct` — l'écart moyen au barreau précédent ;
* `share_improved` — la **part des mondes** où le composant améliore la
  métrique, sa direction étant déclarée par métrique (moins de no-shows est un
  gain, moins de satisfaction n'en est pas un).

C'est `share_improved` qui compte : un composant qui n'aide que la moitié des
mondes n'a pas de contribution robuste, quelle que soit sa moyenne.

In [ ]:
print(ablation.render_mean_table(moyennes.to_dict('records')))

In [ ]:
echelle = moyennes[moyennes['kind'] == 'ladder']

contributions = echelle.pivot_table(
    index='component', columns='metric_label',
    values=['mean_delta_pct', 'share_improved'])
contributions.round(3)

### Les contributions somment-elles à l'écart total ?

Les trois écarts consécutifs doivent reconstituer l'écart Nearest -> BRAM-EV
Full, monde par monde. Un résidu non nul signalerait un barreau manquant ou un
`summary.csv` incohérent.

In [ ]:
pas = detail[(detail['kind'] == 'ladder') &
             (detail['metric'] == 'exact_satisfaction')]

somme = (pas.groupby(['scenario', 'nb_cars'])['delta'].sum()
            .rename('somme_des_contributions'))

extremes = ladder.pivot_table(index=['scenario', 'nb_cars'], columns='method',
                              values='exact_satisfaction', observed=True)
total = (extremes[methods.LADDER[-1]] - extremes[methods.LADDER[0]]
         ).rename('ecart_total')

controle = pd.concat([somme, total], axis=1)
controle['residu'] = (controle['somme_des_contributions']
                      - controle['ecart_total']).abs()
print(f"résidu maximal : {controle['residu'].max():.2e}")
controle.round(6)

## BRAM-EV face aux baselines de référence

Sens de lecture **inverse** de celui de l'échelle : les lignes vont
`baseline -> bramev`, donc `improvement = True` signifie que **BRAM-EV fait
mieux que la baseline**. C'est la question posée à une baseline ; l'échelle,
elle, demande ce qu'un composant apporte.

Les trois baselines ayant le même périmètre d'information que `multistation`,
l'écart mesuré ici ne vient que de la règle de sélection de l'offre.

In [ ]:
reference = moyennes[moyennes['kind'] == 'baseline']

face_a_face = reference.pivot_table(
    index='component', columns='metric_label',
    values=['mean_delta_pct', 'share_improved'])
face_a_face.round(3)

In [ ]:
# Verdict monde par monde, en distinguant les égalités : `improvement` est
# strict, donc un écart nul y compte comme une défaite. Sur une grille peu
# contrainte, deux méthodes rendent souvent *exactement* le même résultat —
# lire cela comme une défaite de BRAM-EV serait faux.
CLES = ['exact_satisfaction', 'rate_abs', 'mean_service_rate']

face = detail[(detail['kind'] == 'baseline') & (detail['metric'].isin(CLES))].copy()
face['issue'] = np.where(face['delta'] == 0, 'nul',
                         np.where(face['improvement'], 'gagné', 'perdu'))

compte = (face.groupby(['from_method', 'metric', 'issue'], observed=True)
              .size().unstack('issue', fill_value=0)
              .reindex(columns=['gagné', 'nul', 'perdu'], fill_value=0))

verdict = (compte.apply(lambda r: f"{r['gagné']}G / {r['nul']}N / {r['perdu']}P", axis=1)
                 .unstack('metric')
                 .rename(index=methods.label))
print("BRAM-EV contre chaque baseline (G gagné / N égalité / P perdu, par monde) :")
print(verdict, end='\n\n')

perdus = compte[compte['perdu'] > compte['gagné']]
if len(perdus):
    for (m, metrique), r in perdus.iterrows():
        print(f"ATTENTION : contre {methods.label(m)}, BRAM-EV perd sur "
              f"{metrique} ({r['perdu']}P contre {r['gagné']}G).")
elif (compte['gagné'] == 0).all():
    print("Aucun écart : les méthodes rendent le même résultat sur tous les "
          "mondes. La grille n'est pas assez contrainte pour les départager — "
          "réduire nb_stations ou augmenter la flotte.")
else:
    print("BRAM-EV n'est battu par aucune baseline sur ces métriques.")

In [ ]:
# Niveaux bruts, sans passer par les écarts : ce que chaque règle de choix
# produit réellement. `min_waiting` doit dominer l'attente, `random_feasible`
# constitue le plancher.
COLONNES = ['exact_satisfaction', 'mean_waiting_time_min',
            'mean_travel_distance_km', 'rate_abs', 'mean_service_rate']

(compare.groupby('method', observed=True)[COLONNES]
        .mean()
        .rename(index=methods.label)
        .round(4))

## Dispersion : la moyenne cache quoi ?

Un écart moyen positif peut n'être porté que par un monde. Voici la
distribution des contributions par composant, sur toutes les tailles de flotte
et tous les scénarios.

In [ ]:
for metrique in ['exact_satisfaction', 'rate_abs', 'mean_service_rate']:
    bloc = detail[(detail['kind'] == 'ladder') & (detail['metric'] == metrique)]
    if bloc.empty:
        continue
    libelle = bloc['metric_label'].iloc[0]
    print(f"\n=== {libelle} — écart au barreau précédent ===")
    stats = (bloc.groupby('component')['delta']
                 .agg(['count', 'min', 'median', 'mean', 'max'])
                 .round(6))
    stats['mondes_améliorés'] = bloc.groupby('component')['improvement'].mean().round(3)
    display(stats)

In [ ]:
# Le composant aide-t-il davantage quand la ressource devient rare ?
# Une contribution qui croît avec la flotte est une contribution qui porte
# sur la congestion, non sur le hasard du tirage.
(detail[(detail['kind'] == 'ladder') &
        (detail['metric'] == 'exact_satisfaction')]
 .pivot_table(index='component', columns='nb_cars', values='delta_pct')
 .round(2))

## Figures

Déjà écrites par le pipeline dans `figures/`. Les régénérer après avoir modifié
`src/pipeline/figures.py` :

```bash
python main.py report --latest
```

In [ ]:
for nom in ['ablation_components', 'scalability']:
    chemin = store.figure_path(nom)
    if chemin.is_file():
        print(chemin.name)
        display(Image(filename=str(chemin)))

for chemin in sorted(store.figures_dir.glob('ablation_ladder_*.png')):
    print(chemin.name)
    display(Image(filename=str(chemin)))

In [ ]:
# Ou reconstruire une figure en mémoire, sans repasser par le disque.
figures.fig_ablation_components(summary.to_dict('records'))

## Garde-fous de lecture

Deux composants ne peuvent structurellement rien montrer sur une campagne trop
courte ou trop peu contrainte. Un `+0.0%` doit alors se lire « mécanisme jamais
sollicité », pas « composant inutile ».

1. **L'adaptation entre stations** ne se déclenche que tous les
   `SOCIETY_UPDATE_INTERVAL` slots (144 par défaut, 12 h).
2. **La recherche multi-stations** n'a d'effet que si la requête atteint
   effectivement plus d'une station, et que la capacité est contrainte : avec
   des bornes libres partout, toute demande est servie de toute façon.

In [ ]:
config = next(store.iter_results())['config']
intervalle = config['society_update_interval']
mises_a_jour = params.total_time // intervalle

print(f"horizon = {params.total_time} slots | "
      f"intervalle d'adaptation = {intervalle} slots")
print(f"-> l'apprentissage collectif se déclenche {mises_a_jour} fois")
if mises_a_jour == 0:
    print("   ATTENTION : jamais déclenché — la contribution mesurée de "
          "l'adaptation est nulle par construction.")

In [ ]:
diffusion = (ladder.groupby('method', observed=True)
                   .agg(offres_par_demande=('mean_offers_per_demand', 'mean'),
                        taux_de_service=('mean_service_rate', 'mean'),
                        demandes_rejetees=('nb_station_level_rejections', 'mean'))
                   .round(3))
print(diffusion, end='\n\n')

gain_offres = (diffusion.loc['multistation', 'offres_par_demande']
               - diffusion.loc['greedy', 'offres_par_demande'])
if gain_offres <= 0:
    print("ATTENTION : la diffusion ne produit pas plus d'offres par demande. "
          "Le rayon de recherche est trop petit devant l'espacement des "
          "stations : le premier barreau n'est pas testé.")
else:
    print(f"La diffusion apporte {gain_offres:+.2f} offre(s) par demande.")

## Santé du run

Un invariant violé ou un diagnostic massif rend toute conclusion suspecte :
à vérifier avant de citer un chiffre.

In [ ]:
print('invariants OK :', bool(summary['invariant_ok'].all()))
print('réservations non résolues :', int(summary['nb_unresolved'].sum()))
print('pannes :', int(summary['nb_breakdowns'].sum()))

vus = set()
for resultat in store.iter_results():
    for message in resultat['behaviors'].get('diagnostics', []):
        if message not in vus:
            vus.add(message)
            print(f"\n[diagnostic] {message}")